In [ ]:
import pandas as pd
from pathlib import Path

CLEAN_DATA = Path("/kaggle/input/datasets/ntquang0410/vntc-dataset")
for name, expected in [("train", 30195), ("val", 3355), ("test", 46167)]:
    df = pd.read_parquet(CLEAN_DATA / f"vntc_{name}_final.parquet")
    status = "✓" if len(df) == expected else "✗"
    print(f"{status} {name}: {len(df)} (expected {expected}), "
          f"file đầu: {df['file'].iloc[0]}")

In [ ]:
!pip install --no-index --find-links=/kaggle/input/datasets/ntquang0410/underthesea/kaggle/working/underthesea_packages underthesea

In [ ]:
# ============================================================
# NOTEBOOK B (v2) — Tokenize + Train trong cùng session
# ============================================================
import os, json, time, re, unicodedata, subprocess
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

# Cài underthesea nếu chưa có (offline-friendly nếu đã có sẵn)
try:
    from underthesea import word_tokenize
except ImportError:
    subprocess.run(["pip", "install", "-q", "underthesea"], check=True)
    from underthesea import word_tokenize

import transformers
print(f"transformers={transformers.__version__}, torch={torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'}")

# ---- ĐƯỜNG DẪN ----
ARTIFACTS   = Path("/kaggle/input/datasets/ntquang0410/vntc-phobert-artifacts")
PHOBERT_DIR = ARTIFACTS / "phobert-base"
CLEAN_DATA  = Path("/kaggle/input/datasets/ntquang0410/vntc-dataset")   # ← thay tên dataset chứa parquet gốc

assert PHOBERT_DIR.exists(), f"Không thấy {PHOBERT_DIR}"
print("\nPhoBERT files:")
for f in PHOBERT_DIR.iterdir():
    print(f"  {f.name}")

# ============================================================
# 1. LOAD & TOKENIZE 3 TẬP
# ============================================================
FOOTER_RE   = re.compile(r"\n[^\n]*\(\d{1,2}/\d{1,2}/\d{4}\)\s*(?:\n.*)*$", re.DOTALL)
EMAIL_RE    = re.compile(r"From:.*?Subject:[^\n]*", re.DOTALL | re.IGNORECASE)
MOJIBAKE_RE = re.compile(r"(\w)&")
WS_RE       = re.compile(r"\s+")

def clean_for_phobert(text):
    if not isinstance(text, str) or not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = FOOTER_RE.sub("", text)
    text = EMAIL_RE.sub("", text)
    text = MOJIBAKE_RE.sub(r"\1 ...", text)
    return WS_RE.sub(" ", text).strip()

def segment_pb(text):
    cleaned = clean_for_phobert(text)
    if not cleaned or len(cleaned) < 10:
        return ""
    try:
        return word_tokenize(cleaned, format="text")
    except Exception:
        return ""

def load_and_tokenize(parquet_path, name, n_jobs=4):
    df = pd.read_parquet(parquet_path)
    print(f"\n>>> {name}: {len(df)} mẫu")
    pb_texts = Parallel(n_jobs=n_jobs, batch_size=200)(
        delayed(segment_pb)(t) for t in tqdm(df["text"].tolist(), desc=name)
    )
    df = df.copy()
    df["pb_text_v2"] = pb_texts
    before = len(df)
    df = df[df["pb_text_v2"].str.len() > 10].reset_index(drop=True)
    if before != len(df):
        print(f"  Dropped {before - len(df)} mẫu rỗng → còn {len(df)}")
    return df

df_train = load_and_tokenize(CLEAN_DATA / "vntc_train_final.parquet", "TRAIN")
df_val   = load_and_tokenize(CLEAN_DATA / "vntc_val_final.parquet",   "VAL")
df_test  = load_and_tokenize(CLEAN_DATA / "vntc_test_final.parquet",  "TEST")

# ---- SANITY CHECK BẮT BUỘC ----
print(f"\n=== KIỂM TRA ===")
print(f"Train = {len(df_train)} (mong đợi ~30195)")
print(f"Val   = {len(df_val)}   (mong đợi ~3355)")
print(f"Test  = {len(df_test)}  (mong đợi ~46167)")

assert 29000 < len(df_train) < 31000, f"Train sai size: {len(df_train)}"
assert 3000  < len(df_val)   < 4000,  f"Val sai size: {len(df_val)}"
assert 44000 < len(df_test)  < 47000, f"Test sai size: {len(df_test)}"
print("✓ Sizes OK")

# Lưu lại tokenized vào /kaggle/working để các runs sau khỏi tokenize lại
TOK_OUT = Path("/kaggle/working/tokenized")
TOK_OUT.mkdir(exist_ok=True)
df_train.to_parquet(TOK_OUT / "train.parquet", index=False)
df_val.to_parquet(TOK_OUT / "val.parquet", index=False)
df_test.to_parquet(TOK_OUT / "test.parquet", index=False)
print(f"Saved tokenized to {TOK_OUT}")

# ============================================================
# 2. TRAIN PhoBERT
# ============================================================
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback,
)

RUN_NAME   = "run1_trunc256_ce"
MAX_LEN    = 256
BATCH_SIZE = 32
EPOCHS     = 4
LR         = 2e-5
SEED       = 42

OUT_DIR = Path("/kaggle/working/runs") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED); np.random.seed(SEED)

le = LabelEncoder()
df_train["label"] = le.fit_transform(df_train["class"])
df_val["label"]   = le.transform(df_val["class"])
df_test["label"]  = le.transform(df_test["class"])

tokenizer = AutoTokenizer.from_pretrained(str(PHOBERT_DIR), use_fast=False)

def to_hf(df):
    ds = Dataset.from_pandas(df[["pb_text_v2", "label"]], preserve_index=False)
    return ds.map(
        lambda batch: tokenizer(batch["pb_text_v2"], truncation=True, max_length=MAX_LEN),
        batched=True, remove_columns=["pb_text_v2"],
    )

ds_train = to_hf(df_train)
ds_val   = to_hf(df_val)
ds_test  = to_hf(df_test)
print(f"\nHF Datasets: train={len(ds_train)}, val={len(ds_val)}, test={len(ds_test)}")

# ---- KIỂM TRA LẦN CUỐI TRƯỚC KHI TRAIN ----
assert len(ds_test) > 40000, f"ds_test sai size: {len(ds_test)}"

collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    str(PHOBERT_DIR), num_labels=len(le.classes_),
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":    accuracy_score(labels, preds),
        "f1_macro":    f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

ta_kwargs = dict(
    output_dir=str(OUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR, weight_decay=0.01, warmup_ratio=0.1,
    save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True,
    logging_steps=100, fp16=True, seed=SEED, report_to="none",
)
try:
    args = TrainingArguments(**ta_kwargs, eval_strategy="epoch")
except TypeError:
    args = TrainingArguments(**ta_kwargs, evaluation_strategy="epoch")

# Compat tokenizer / processing_class
common_kwargs = dict(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
try:
    trainer = Trainer(**common_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = Trainer(**common_kwargs, tokenizer=tokenizer)

t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f"\n>>> Train time: {train_time/60:.1f} phút")

# ============================================================
# 3. EVAL TRÊN TEST (đúng tập!)
# ============================================================
print(f"\n=== EVAL TRÊN TEST ({len(ds_test)} mẫu) ===")
test_out = trainer.predict(ds_test)
y_pred = np.argmax(test_out.predictions, axis=-1)
y_true = test_out.label_ids

# Sanity: phải có ~46167 mẫu
assert len(y_true) > 40000, f"y_true sai size: {len(y_true)} — KHÔNG PHẢI test!"

test_acc = accuracy_score(y_true, y_pred)
test_f1  = f1_score(y_true, y_pred, average="macro")
test_f1w = f1_score(y_true, y_pred, average="weighted")
print(f"Acc={test_acc:.4f}, F1-macro={test_f1:.4f}, F1-weighted={test_f1w:.4f}\n")
print(classification_report(y_true, y_pred, target_names=le.classes_, digits=4))

# ============================================================
# 4. LƯU ARTIFACTS
# ============================================================
trainer.save_model(str(OUT_DIR / "best"))
tokenizer.save_pretrained(str(OUT_DIR / "best"))
np.save(OUT_DIR / "test_pred.npy", y_pred)
np.save(OUT_DIR / "test_true.npy", y_true)
np.save(OUT_DIR / "test_logits.npy", test_out.predictions)

report_dict = classification_report(
    y_true, y_pred, target_names=le.classes_, digits=4, output_dict=True
)
metrics = {
    "run_name": RUN_NAME,
    "config": {"model":"phobert-base","max_len":MAX_LEN,"batch":BATCH_SIZE,
               "lr":LR,"epochs":EPOCHS,"strategy":"truncation_256","loss":"ce"},
    "train_time_min": round(train_time/60, 2),
    "n_train": len(ds_train), "n_val": len(ds_val), "n_test": len(ds_test),
    "test_accuracy":    round(test_acc, 4),
    "test_f1_macro":    round(test_f1, 4),
    "test_f1_weighted": round(test_f1w, 4),
    "per_class": report_dict,
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f"\nSaved to {OUT_DIR}")

In [ ]:
# Xem lịch sử training
print("=== TRAINING HISTORY ===")
for log in trainer.state.log_history:
    if "eval_f1_macro" in log:
        print(f"Epoch {log['epoch']:.0f}: "
              f"eval_loss={log.get('eval_loss', 0):.4f}, "
              f"eval_f1={log['eval_f1_macro']:.4f}, "
              f"eval_acc={log.get('eval_accuracy', 0):.4f}")

print(f"\nBest epoch: {trainer.state.best_metric}")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Global steps: {trainer.state.global_step}")
print(f"Max steps: {trainer.state.max_steps}")

In [ ]:
# Verify đã lưu
import json
from pathlib import Path
RUN1_DIR = Path("/kaggle/working/runs/run1_trunc256_ce")
with open(RUN1_DIR / "metrics.json") as f:
    m = json.load(f)
print(json.dumps(m["per_class"]["macro avg"], indent=2))
print(f"Train time: {m['train_time_min']} min")
print(f"Test F1-macro: {m['test_f1_macro']}")

In [ ]:
# ============================================================
# RUN 2: PhoBERT-base + head+tail 256 + Cross-Entropy
# ============================================================
from pathlib import Path

RUN_NAME   = "run2_headtail_ce"
HEAD_LEN   = 128
TAIL_LEN   = 126   # 128 + 126 + 2 special tokens ([CLS], [SEP]) = 256

OUT_DIR = Path("/kaggle/working/runs") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Tokenize MỚI: head+tail strategy
def head_tail_tokenize(examples):
    """Cho mỗi văn bản: lấy HEAD_LEN token đầu + TAIL_LEN token cuối."""
    texts = examples["pb_text_v2"]
    all_input_ids = []
    all_attention = []
    
    for text in texts:
        # Tokenize without truncation
        tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
        
        if len(tokens) <= HEAD_LEN + TAIL_LEN:
            # Văn bản ngắn, dùng nguyên
            ids = tokens
        else:
            # Lấy đầu + cuối
            ids = tokens[:HEAD_LEN] + tokens[-TAIL_LEN:]
        
        # Thêm [CLS] và [SEP]
        ids = [tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]
        att = [1] * len(ids)
        
        all_input_ids.append(ids)
        all_attention.append(att)
    
    return {"input_ids": all_input_ids, "attention_mask": all_attention}

# Re-create datasets với strategy mới
def to_hf_headtail(df):
    ds = Dataset.from_pandas(df[["pb_text_v2", "label"]], preserve_index=False)
    return ds.map(head_tail_tokenize, batched=True, batch_size=200,
                  remove_columns=["pb_text_v2"])

print("Tokenizing với head+tail strategy...")
ds_train = to_hf_headtail(df_train)
ds_val   = to_hf_headtail(df_val)
ds_test  = to_hf_headtail(df_test)

# Verify
sample = ds_train[0]
print(f"Sample input length: {len(sample['input_ids'])} (max 256)")
assert len(sample['input_ids']) <= 256

# Reload model từ pretrained (fresh weights)
model = AutoModelForSequenceClassification.from_pretrained(
    str(PHOBERT_DIR), num_labels=len(le.classes_),
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
)

# Cùng training args
ta_kwargs = dict(
    output_dir=str(OUT_DIR / "checkpoints"),
    num_train_epochs=4, per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True,
    logging_steps=100, fp16=True, seed=42, report_to="none",
)
try:
    args = TrainingArguments(**ta_kwargs, eval_strategy="epoch")
except TypeError:
    args = TrainingArguments(**ta_kwargs, evaluation_strategy="epoch")

common_kwargs = dict(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
try:
    trainer = Trainer(**common_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = Trainer(**common_kwargs, tokenizer=tokenizer)

t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f"Train time: {train_time/60:.1f} phút")

# Eval test
test_out = trainer.predict(ds_test)
y_pred = np.argmax(test_out.predictions, axis=-1)
y_true = test_out.label_ids
assert len(y_true) == 46167

test_acc = accuracy_score(y_true, y_pred)
test_f1  = f1_score(y_true, y_pred, average="macro")
test_f1w = f1_score(y_true, y_pred, average="weighted")
print(f"\nAcc={test_acc:.4f}, F1-macro={test_f1:.4f}, F1-weighted={test_f1w:.4f}\n")
print(classification_report(y_true, y_pred, target_names=le.classes_, digits=4))

# Lưu
trainer.save_model(str(OUT_DIR / "best"))
np.save(OUT_DIR / "test_pred.npy", y_pred)

report_dict = classification_report(y_true, y_pred, target_names=le.classes_, digits=4, output_dict=True)
metrics = {
    "run_name": RUN_NAME,
    "config": {"strategy":"head_tail", "head_len":HEAD_LEN, "tail_len":TAIL_LEN, "loss":"ce"},
    "train_time_min": round(train_time/60, 2),
    "test_accuracy": round(test_acc, 4),
    "test_f1_macro": round(test_f1, 4),
    "test_f1_weighted": round(test_f1w, 4),
    "per_class": report_dict,
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f"\nSaved → {OUT_DIR}")

In [ ]:
print("=== TRAINING HISTORY RUN 2 ===")
for log in trainer.state.log_history:
    if "eval_f1_macro" in log:
        print(f"Epoch {log['epoch']:.0f}: loss={log.get('eval_loss', 0):.4f}, "
              f"f1={log['eval_f1_macro']:.4f}, acc={log.get('eval_accuracy', 0):.4f}")

In [ ]:
# ============================================================
# RUN 3: head+tail 256 + Cross-Entropy với CLASS WEIGHTS
# ============================================================
import torch.nn as nn

RUN_NAME = "run3_headtail_weighted"
OUT_DIR = Path("/kaggle/working/runs") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Tính class weights ----
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(le.classes_)),
    y=df_train["label"].values
)
print("Class weights:")
for cls, w in zip(le.classes_, class_weights):
    n = (df_train["label"] == le.transform([cls])[0]).sum()
    print(f"  {cls:20s}: n={n:>5}, weight={w:.3f}")

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).cuda()

# ---- Custom Trainer với weighted loss ----
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ---- Re-tokenize với head+tail (như Run 2) ----
ds_train = to_hf_headtail(df_train)
ds_val   = to_hf_headtail(df_val)
ds_test  = to_hf_headtail(df_test)

# ---- Fresh model ----
model = AutoModelForSequenceClassification.from_pretrained(
    str(PHOBERT_DIR), num_labels=len(le.classes_),
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
)

# ---- TrainingArguments giống Run 2 ----
ta_kwargs = dict(
    output_dir=str(OUT_DIR / "checkpoints"),
    num_train_epochs=4, per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True,
    logging_steps=100, fp16=True, seed=42, report_to="none",
)
try:
    args = TrainingArguments(**ta_kwargs, eval_strategy="epoch")
except TypeError:
    args = TrainingArguments(**ta_kwargs, evaluation_strategy="epoch")

common_kwargs = dict(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights_tensor,    # ← TRUYỀN CLASS WEIGHTS
)
try:
    trainer = WeightedTrainer(**common_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = WeightedTrainer(**common_kwargs, tokenizer=tokenizer)

t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f"Train time: {train_time/60:.1f} phút")

# ---- Eval test ----
test_out = trainer.predict(ds_test)
y_pred = np.argmax(test_out.predictions, axis=-1)
y_true = test_out.label_ids
assert len(y_true) == 46167

test_acc = accuracy_score(y_true, y_pred)
test_f1  = f1_score(y_true, y_pred, average="macro")
test_f1w = f1_score(y_true, y_pred, average="weighted")
print(f"\nAcc={test_acc:.4f}, F1-macro={test_f1:.4f}, F1-weighted={test_f1w:.4f}\n")
print(classification_report(y_true, y_pred, target_names=le.classes_, digits=4))

# Lưu
trainer.save_model(str(OUT_DIR / "best"))
np.save(OUT_DIR / "test_pred.npy", y_pred)
report_dict = classification_report(y_true, y_pred, target_names=le.classes_, digits=4, output_dict=True)
metrics = {
    "run_name": RUN_NAME,
    "config": {"strategy":"head_tail", "loss":"weighted_ce", 
               "weights": class_weights.tolist()},
    "train_time_min": round(train_time/60, 2),
    "test_accuracy": round(test_acc, 4),
    "test_f1_macro": round(test_f1, 4),
    "test_f1_weighted": round(test_f1w, 4),
    "per_class": report_dict,
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f"\nSaved → {OUT_DIR}")

In [ ]:
# Kiểm tra weights có thực sự được áp dụng không
print("Class weights tensor:", trainer.class_weights)
print("Device:", trainer.class_weights.device if trainer.class_weights is not None else "None")

# Test compute_loss có nhận weights không
import torch
sample = next(iter(trainer.get_train_dataloader()))
sample = {k: v.cuda() for k, v in sample.items()}
labels = sample.pop("labels")

# Standard loss
out = model(**sample)
standard_loss = torch.nn.CrossEntropyLoss()(out.logits, labels)

# Weighted loss
weighted_loss = torch.nn.CrossEntropyLoss(weight=trainer.class_weights)(out.logits, labels)

print(f"Standard CE loss: {standard_loss.item():.4f}")
print(f"Weighted CE loss: {weighted_loss.item():.4f}")
print(f"Difference: {(weighted_loss - standard_loss).item():.4f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

classes = list(le.classes_)
cm = confusion_matrix(y_true, y_pred, normalize="true")

# In ra Đời sống bị nhầm sang đâu
ds_idx = classes.index("Doi song")
print(f"Đời sống bị nhầm sang:")
for i, cls in enumerate(classes):
    if i != ds_idx and cm[ds_idx, i] > 0.02:
        print(f"  {cls}: {cm[ds_idx, i]:.1%}")

# Vẽ heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Run 3 (PhoBERT + head+tail + weighted CE)")
plt.tight_layout()
plt.savefig("/kaggle/working/runs/run3_headtail_weighted/confusion.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# NOTEBOOK C — RUN 4: PhoBERT-LARGE + head+tail + CE
# ============================================================
import os, json, time, re, unicodedata, subprocess
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback,
)

try:
    from underthesea import word_tokenize
except ImportError:
    subprocess.run(["pip", "install", "-q", "underthesea"], check=True)
    from underthesea import word_tokenize

print(f"CUDA: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB")

# ---- ĐƯỜNG DẪN ----
ARTIFACTS = Path("/kaggle/input/datasets/ntquang0410/vntc-phobert-artifacts")
PHOBERT_LARGE_DIR = ARTIFACTS / "phobert-large"
CLEAN_DATA = Path("/kaggle/input/datasets/ntquang0410/vntc-dataset")   # ← THAY CHO ĐÚNG

assert PHOBERT_LARGE_DIR.exists(), f"Không thấy {PHOBERT_LARGE_DIR} — kiểm tra dataset đã update chưa"
print(f"\nPhoBERT-large files:")
for f in sorted(PHOBERT_LARGE_DIR.iterdir()):
    print(f"  {f.name}: {f.stat().st_size/1024/1024:.1f}MB")

# ============================================================
# 1. LOAD + TOKENIZE (giữ dấu câu)
# ============================================================
FOOTER_RE   = re.compile(r"\n[^\n]*\(\d{1,2}/\d{1,2}/\d{4}\)\s*(?:\n.*)*$", re.DOTALL)
EMAIL_RE    = re.compile(r"From:.*?Subject:[^\n]*", re.DOTALL | re.IGNORECASE)
MOJIBAKE_RE = re.compile(r"(\w)&")
WS_RE       = re.compile(r"\s+")

def clean_for_phobert(text):
    if not isinstance(text, str) or not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = FOOTER_RE.sub("", text)
    text = EMAIL_RE.sub("", text)
    text = MOJIBAKE_RE.sub(r"\1 ...", text)
    return WS_RE.sub(" ", text).strip()

def segment_pb(text):
    cleaned = clean_for_phobert(text)
    if not cleaned or len(cleaned) < 10:
        return ""
    try:
        return word_tokenize(cleaned, format="text")
    except Exception:
        return ""

def load_and_tokenize(parquet_path, name, n_jobs=4):
    df = pd.read_parquet(parquet_path)
    print(f"\n>>> {name}: {len(df)} mẫu")
    pb_texts = Parallel(n_jobs=n_jobs, batch_size=200)(
        delayed(segment_pb)(t) for t in tqdm(df["text"].tolist(), desc=name)
    )
    df = df.copy()
    df["pb_text_v2"] = pb_texts
    before = len(df)
    df = df[df["pb_text_v2"].str.len() > 10].reset_index(drop=True)
    if before != len(df):
        print(f"  Drop {before - len(df)} rỗng → còn {len(df)}")
    return df

df_train = load_and_tokenize(CLEAN_DATA / "vntc_train_final.parquet", "TRAIN")
df_val   = load_and_tokenize(CLEAN_DATA / "vntc_val_final.parquet",   "VAL")
df_test  = load_and_tokenize(CLEAN_DATA / "vntc_test_final.parquet",  "TEST")

# Sanity check
assert 29000 < len(df_train) < 31000, f"Train sai: {len(df_train)}"
assert 3000  < len(df_val)   < 4000,  f"Val sai: {len(df_val)}"
assert 44000 < len(df_test)  < 47000, f"Test sai: {len(df_test)}"
print(f"\n✓ Sizes OK: Train={len(df_train)}, Val={len(df_val)}, Test={len(df_test)}")

# ============================================================
# 2. SETUP MODEL + TOKENIZER
# ============================================================
RUN_NAME = "run4_large_headtail_ce"
OUT_DIR = Path("/kaggle/working/runs") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

le = LabelEncoder()
df_train["label"] = le.fit_transform(df_train["class"])
df_val["label"]   = le.transform(df_val["class"])
df_test["label"]  = le.transform(df_test["class"])
print(f"\nLabels ({len(le.classes_)}): {list(le.classes_)}")

tokenizer = AutoTokenizer.from_pretrained(str(PHOBERT_LARGE_DIR), use_fast=False)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# ============================================================
# 3. HEAD+TAIL TOKENIZATION
# ============================================================
HEAD_LEN, TAIL_LEN = 128, 126   # +2 special tokens = 256

def head_tail_tokenize(examples):
    texts = examples["pb_text_v2"]
    all_input_ids, all_attention = [], []
    for text in texts:
        tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(tokens) <= HEAD_LEN + TAIL_LEN:
            ids = tokens
        else:
            ids = tokens[:HEAD_LEN] + tokens[-TAIL_LEN:]
        ids = [tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]
        all_input_ids.append(ids)
        all_attention.append([1] * len(ids))
    return {"input_ids": all_input_ids, "attention_mask": all_attention}

def to_hf(df):
    ds = Dataset.from_pandas(df[["pb_text_v2", "label"]], preserve_index=False)
    return ds.map(head_tail_tokenize, batched=True, batch_size=200,
                  remove_columns=["pb_text_v2"])

print("\nTokenizing với head+tail...")
ds_train = to_hf(df_train)
ds_val   = to_hf(df_val)
ds_test  = to_hf(df_test)

# Verify
sample = ds_train[0]
print(f"Sample length: {len(sample['input_ids'])} (max 256)")
assert len(sample['input_ids']) <= 256

collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================================
# 4. MODEL
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    str(PHOBERT_LARGE_DIR), num_labels=len(le.classes_),
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel params: {n_params/1e6:.0f}M")

# ============================================================
# 5. TRAINING
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":    accuracy_score(labels, preds),
        "f1_macro":    f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

ta_kwargs = dict(
    output_dir=str(OUT_DIR / "checkpoints"),
    num_train_epochs=4,
    per_device_train_batch_size=16,         # large model
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,           # effective batch=32
    learning_rate=1e-5,                      # large nhạy hơn, giảm từ 2e-5
    weight_decay=0.01, warmup_ratio=0.1,
    save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True,
    logging_steps=100, fp16=True, seed=SEED, report_to="none",
)
try:
    args = TrainingArguments(**ta_kwargs, eval_strategy="epoch")
except TypeError:
    args = TrainingArguments(**ta_kwargs, evaluation_strategy="epoch")

common_kwargs = dict(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
try:
    trainer = Trainer(**common_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = Trainer(**common_kwargs, tokenizer=tokenizer)

t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f"\n>>> Train time: {train_time/60:.1f} phút")

# ============================================================
# 6. EVAL TRÊN TEST
# ============================================================
print(f"\n=== EVAL TRÊN TEST ({len(ds_test)} mẫu) ===")
test_out = trainer.predict(ds_test)
y_pred = np.argmax(test_out.predictions, axis=-1)
y_true = test_out.label_ids
assert len(y_true) == 46167, f"Test size sai: {len(y_true)}"

test_acc = accuracy_score(y_true, y_pred)
test_f1  = f1_score(y_true, y_pred, average="macro")
test_f1w = f1_score(y_true, y_pred, average="weighted")
print(f"\nAcc={test_acc:.4f}, F1-macro={test_f1:.4f}, F1-weighted={test_f1w:.4f}\n")
print(classification_report(y_true, y_pred, target_names=le.classes_, digits=4))

# Training history
print("\n=== TRAINING HISTORY ===")
for log in trainer.state.log_history:
    if "eval_f1_macro" in log:
        print(f"Epoch {log['epoch']:.0f}: loss={log.get('eval_loss', 0):.4f}, "
              f"f1={log['eval_f1_macro']:.4f}, acc={log.get('eval_accuracy', 0):.4f}")

# ============================================================
# 7. LƯU
# ============================================================
trainer.save_model(str(OUT_DIR / "best"))
np.save(OUT_DIR / "test_pred.npy", y_pred)
np.save(OUT_DIR / "test_true.npy", y_true)

report_dict = classification_report(y_true, y_pred, target_names=le.classes_, digits=4, output_dict=True)
metrics = {
    "run_name": RUN_NAME,
    "config": {"model":"phobert-large", "params_M": round(n_params/1e6),
               "strategy":"head_tail", "head":HEAD_LEN, "tail":TAIL_LEN,
               "loss":"ce", "batch":16, "grad_accum":2, "lr":1e-5, "epochs":4},
    "train_time_min": round(train_time/60, 2),
    "n_train": len(ds_train), "n_val": len(ds_val), "n_test": len(ds_test),
    "test_accuracy":    round(test_acc, 4),
    "test_f1_macro":    round(test_f1, 4),
    "test_f1_weighted": round(test_f1w, 4),
    "per_class": report_dict,
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"\nSaved → {OUT_DIR}")
print(json.dumps({k:v for k,v in metrics.items() if k != "per_class"}, indent=2, ensure_ascii=False))